<a href="https://colab.research.google.com/github/yashsati072-create/Movie-Recommendation-System/blob/main/Movie-Recommendation-System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
os.environ['KAGGLE_API_TOKEN'] = ''
!pip install -q kaggle
!kaggle datasets download -d rounakbanik/the-movies-dataset

Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0
the-movies-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
with zipfile.ZipFile('the-movies-dataset.zip','r') as zip_ref:
  zip_ref.extractall('dataset')

base = "dataset"
credit = pd.read_csv(f"{base}/credits.csv")
keyword=pd.read_csv(f"{base}/keywords.csv")
link=pd.read_csv(f"{base}/links.csv")
link_small=pd.read_csv(f"{base}/links_small.csv")
movie_da=pd.read_csv(f"{base}/movies_metadata.csv")
rating=pd.read_csv(f"{base}/ratings.csv")
rating_sm=pd.read_csv(f"{base}/ratings_small.csv")

/tmp/ipykernel_1486/3571181893.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movie_da=pd.read_csv(f"{base}/movies_metadata.csv")


# Data Inspection


In [ ]:
credit.head(10)
print(f"Shape:{credit.shape}")
print(f"Columns:{list(credit.columns)}")
print('\n Missing Values:')
print(credit.isnull().sum())
print(f"Duplicate:{credit.duplicated().sum()}")



Shape:(45476, 3)
Columns:['cast', 'crew', 'id']

 Missing Values:
cast    0
crew    0
id      0
dtype: int64
Duplicate:37


In [ ]:
credit=credit.drop_duplicates()

#Keyword Data Inespection

In [ ]:
keyword.head(10)
print(f"Shape:{keyword.shape}")
print(f"Columns:{list(keyword.columns)}")
print('\n Missing Values:')
print(keyword.isnull().sum())
print(f"Duplicate:{keyword.duplicated().sum()}")
keyword=keyword.drop_duplicates()



Shape:(46419, 2)
Columns:['id', 'keywords']

 Missing Values:
id          0
keywords    0
dtype: int64
Duplicate:987


#Links Data inspection

In [ ]:
link.head(10)
print(f"Shape:{link.shape}"
)
print(f"Columns:{list(link.columns)}")
print('\n Missing Values:')
print(link.isnull().sum())
print(f"Duplicate:{link.dupldicated().sum()})")





Shape:(45843, 3)
Columns:['movieId', 'imdbId', 'tmdbId']

 Missing Values:
movieId      0
imdbId       0
tmdbId     219
dtype: int64


AttributeError: 'DataFrame' object has no attribute 'dupldicated'

In [ ]:
link.head(10)
link['tmdbId'].fillna(link["tmdbId"].median(),inplace=True)

#Links small Data inspection


In [ ]:
link_small.head(10)
print(f"Shape:{link_small.shape}"
)
print(f"Columns:{list(link_small.columns)}")
print('\n Missing Values:')
print(link_small.isnull().sum())
print(f"Duplicate:{link_small.duplicated().sum()})")

In [ ]:

link_small['tmdbId'].fillna(link_small["tmdbId"].median(),inplace=True)

# movie metadata data inspection


In [ ]:
movie_da.head(10)
print(f"Shape:{movie_da.shape}"
)
print(f"Columns:{list(movie_da.columns)}")
print('\n Missing Values:')
print(movie_da.isnull().sum())
print(f"Duplicate:{movie_da.duplicated().sum()})")

Shape:(45453, 24)
Columns:['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']

 Missing Values:
adult                        0
belongs_to_collection    40959
budget                       0
genres                       0
homepage                 37673
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  250

In [ ]:
movie_da.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [ ]:
# Remove extra spaces
movie_da['belongs_to_collection'] = movie_da['belongs_to_collection'].astype(str).str.strip()

In [ ]:
movie_da=movie_da.drop_duplicates()

movie_da['belongs_to_collection'].fillna(movie_da['belongs_to_collection'].mode()[0],inplace=True)

/tmp/ipykernel_1486/3171952200.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movie_da['belongs_to_collection'].fillna(movie_da['belongs_to_collection'].mode()[0],inplace=True)
